# Fovea backend on Colab

Runs the Fovea **backend API** (`python -m backend serve`, `MODE=local`) on this Colab GPU and makes it reachable
from the internet through a **Cloudflare quick tunnel** (no account needed). Jobs run on this runtime's GPU.

1. `Runtime → Change runtime type → GPU` (T4 / L4 / A100).
2. Run the cells top to bottom. Cell 5 prints the **public URL** and the **API key**.
3. Point your frontend at that URL (send the key as the `X-API-Key` header).

> ⚠️ The backend only lives as long as this runtime. The tunnel URL changes on every start.
> Cloudflare quick tunnels cap request bodies at ~100 MB, so uploads larger than that fail.

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/atharva-google/fovea.git"  # @param {type:"string"}
BRANCH = "perf/parallel-stages"  # @param {type:"string"}
# Browser origins of your frontend (comma separated) - needed for uploads/downloads from the browser.
ALLOWED_ORIGINS = "http://localhost:3000,http://127.0.0.1:3000"  # @param {type:"string"}
USE_DRIVE = False  # @param {type:"boolean"}
DOWNLOAD_TRANSCRIBE_WEIGHTS = False  # @param {type:"boolean"}
DEMO_MODE = False  # @param {type:"boolean"}
MAX_CONCURRENT_JOBS = 1  # @param {type:"integer"}
PORT = 8000

## 2. GPU check, clone & install
If the repo is private, add a Colab secret named `GITHUB_TOKEN` (🔑 icon in the left bar).

In [ ]:
import os, subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout or "⚠️ No GPU - jobs will run on CPU (slow)")

url = REPO_URL
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if token:
        url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass

os.chdir("/content")
if not os.path.exists("/content/fovea"):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", url, "/content/fovea"], check=True)
else:
    subprocess.run(["git", "-C", "/content/fovea", "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", "/content/fovea", "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
os.chdir("/content/fovea")
!git log --oneline -1
!pip install -q -r requirements.txt -r backend/requirements.txt

> If pip asks for a runtime restart (e.g. numpy changed): `Runtime → Restart session`, then run cells 1-2 again.

## 3. Storage (optional Google Drive) & model weights

In [ ]:
DATA_DIR = "/content/fovea_data"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["FOVEA_WEIGHTS_DIR"] = "/content/drive/MyDrive/fovea_weights"
    DATA_DIR = "/content/drive/MyDrive/fovea_data"   # uploads, results and job records survive restarts
os.makedirs(DATA_DIR, exist_ok=True)

!python -m fovea download {"--all" if DOWNLOAD_TRANSCRIBE_WEIGHTS else ""}
!python -m fovea weights

## 4. Start the Cloudflare tunnel
Started first because the backend needs its public URL (it is used in upload/download links).

In [ ]:
import re, time

if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /usr/local/bin/cloudflared

if "tunnel_proc" in globals() and tunnel_proc.poll() is None:
    tunnel_proc.terminate()
tunnel_log = open("/content/cloudflared.log", "w")
tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://localhost:{PORT}"],
                               stdout=tunnel_log, stderr=subprocess.STDOUT)
PUBLIC_URL = None
for _ in range(60):
    found = [u for u in re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
             if not u.startswith("https://api.")]
    if found:
        PUBLIC_URL = found[0]
        break
    time.sleep(1)
assert PUBLIC_URL, "Tunnel did not start - see /content/cloudflared.log"
print("Public URL:", PUBLIC_URL)

## 5. Start the backend

In [ ]:
import secrets, requests

if "API_KEY" not in globals():
    API_KEY = secrets.token_urlsafe(24)

env = {
    **os.environ,
    "MODE": "local",
    "DATA_DIR": DATA_DIR,
    "PORT": str(PORT),
    "PUBLIC_URL": PUBLIC_URL,
    "ALLOWED_ORIGINS": ALLOWED_ORIGINS,
    "AUTH_MODE": "api_key",
    "API_KEY": API_KEY,
    "DEMO_MODE": str(DEMO_MODE).lower(),
    "MAX_CONCURRENT_JOBS": str(MAX_CONCURRENT_JOBS),
}
if "backend_proc" in globals() and backend_proc.poll() is None:
    backend_proc.terminate(); backend_proc.wait()
backend_log = open("/content/backend.log", "w")
backend_proc = subprocess.Popen(["python", "-m", "backend", "serve", "--host", "0.0.0.0", "--port", str(PORT)],
                                cwd="/content/fovea", env=env, stdout=backend_log, stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        if requests.get(f"http://localhost:{PORT}/healthz", timeout=2).ok:
            break
    except requests.RequestException:
        pass
    if backend_proc.poll() is not None:
        raise RuntimeError("Backend exited:\n" + open("/content/backend.log").read()[-3000:])
    time.sleep(1)

print("Through the tunnel:", requests.get(f"{PUBLIC_URL}/healthz", timeout=30).json())
print(f"\n  API URL : {PUBLIC_URL}\n  Docs    : {PUBLIC_URL}/docs\n  API key : {API_KEY}   (header X-API-Key)")

## 6. (Optional) End-to-end test: upload a short video through the public URL

In [ ]:
from google.colab import files
uploaded = files.upload()
VIDEO = list(uploaded)[0]

H = {"X-API-Key": API_KEY}
USER = "colab-test"
r = requests.post(f"{PUBLIC_URL}/v1/uploads", headers=H,
                  json={"user_id": USER, "filename": VIDEO, "size_bytes": os.path.getsize(VIDEO)})
r.raise_for_status()
job_id, up = r.json()["job_id"], r.json()["upload"]
with open(VIDEO, "rb") as f:
    requests.request(up["method"], up["url"], data=f, headers=up.get("headers") or {}).raise_for_status()
r = requests.post(f"{PUBLIC_URL}/v1/jobs/{job_id}/start", headers=H,
                  json={"user_id": USER, "targets": ["9:16"], "options": {}})
r.raise_for_status()
print("started", job_id)

def describe(job):
    parts = []
    for name, st in (job.get("steps") or {}).items():
        text = st.get("status") or "pending"
        stage = st.get("stage")
        if stage:
            pos = f"{stage['index']}/{stage['count']} " if stage.get("index") else ""
            cnt = f" {stage['done']}/{stage['total']} {stage['unit']}" if stage.get("total") else ""
            det = f" - {stage['detail']}" if stage.get("detail") else ""
            text += f" [{pos}{stage['name']}{cnt}, {stage.get('elapsed_s') or 0}s{det}]"
        parts.append(f"{name}: {text}")
    return " | ".join(parts) or "waiting to start"

last = None
while True:
    job = requests.get(f"{PUBLIC_URL}/v1/jobs/{job_id}", headers=H, params={"user_id": USER}).json()
    line = f"{job['status']:<9} {describe(job)}"
    if line != last:
        print(time.strftime("%H:%M:%S"), line)
        last = line
    if job["status"] in ("rendered", "complete", "failed"):
        break
    time.sleep(3)

print("\nTime per stage (s):")
for name, st in (job.get("steps") or {}).items():
    print(f"  {name:<7}", st.get("stage_times"))
print("\nWhat the visual step found:", (job.get("steps") or {}).get("visual", {}).get("summary"))
if job.get("error"):
    print("ERROR:", job["error"])

In [ ]:
from fovea.utils.notebook import show_video
for label, link in job.get("outputs", {}).items():
    out = f"/content/{job_id}_{label.replace(':', 'x')}.mp4"
    open(out, "wb").write(requests.get(link).content)
    print(label, "->", out)
    show_video(out)

## 7. Logs / keep alive
Shows the latest backend log lines. Keep this tab open: Colab stops idle runtimes.

In [ ]:
!tail -n 40 /content/backend.log

## 8. Stop

In [ ]:
for p in ("backend_proc", "tunnel_proc"):
    if p in globals() and globals()[p].poll() is None:
        globals()[p].terminate()
print("stopped")